In [17]:
import os
os.chdir(r"C:\Users\14731\desktop\Capstone\Wifi-indoor-localization-with-rtt-and-csi\DL")
print("当前工作目录:", os.getcwd())

当前工作目录: C:\Users\14731\desktop\Capstone\Wifi-indoor-localization-with-rtt-and-csi\DL


In [18]:
import pandas as pd
import numpy as np
import ast

def align_csi_ftm_from_csv(ftm_csv, csi_csv, window_ms=100):
    """
    对齐 CSI 和 FTM 数据
    支持两种模式：
        - 时间加权平均（已注释）
        - 匹配最近的一个 CSI（当前使用）
    返回：
        - csi_list: 每条 CSI 的 I/Q 数据
        - rssi_list: 对应的 RSSI
        - rtt_list: 对应的 RTT (ns)
    window_ms: 对齐时间窗口，单位毫秒
    """
    # 读取 CSV
    ftm_df = pd.read_csv(ftm_csv)
    csi_df = pd.read_csv(csi_csv)

    # 统一列名
    ftm_df.rename(columns=lambda x: x.strip().lower(), inplace=True)
    csi_df.rename(columns=lambda x: x.strip().lower(), inplace=True)

    # 转换 timestamp 为 datetime
    ftm_df['timestamp'] = pd.to_datetime(ftm_df['timestamp'], errors='coerce')
    csi_df['timestamp'] = pd.to_datetime(csi_df['timestamp'], errors='coerce')

    # 删除无法解析的行
    ftm_df.dropna(subset=['timestamp'], inplace=True)
    csi_df.dropna(subset=['timestamp'], inplace=True)

    # 初始化输出列表
    csi_list = []
    rssi_list = []
    rtt_list = []

    window = pd.Timedelta(milliseconds=window_ms)

    for _, ftm_row in ftm_df.iterrows():
        ftm_time = ftm_row['timestamp']

        # 找到时间窗口内的 CSI
        mask = (csi_df['timestamp'] >= ftm_time - window) & \
               (csi_df['timestamp'] <= ftm_time + window)
        csi_window = csi_df[mask]

        if len(csi_window) == 0:
            continue

        # -------------------------
        # ✅ 方法1：匹配最近的一个 CSI（当前启用）
        # -------------------------
        time_diffs = np.abs(csi_window['timestamp'] - ftm_time)
        nearest_idx = time_diffs.idxmin()
        nearest_row = csi_window.loc[nearest_idx]

        if isinstance(nearest_row['data'], str):
            csi_avg = np.array(ast.literal_eval(nearest_row['data']), dtype=np.float32)
        else:
            csi_avg = np.array(nearest_row['data'], dtype=np.float32)

        # -------------------------
        # ❌ 方法2：时间差加权平均（已注释）
        # -------------------------
        """
        times = csi_window['timestamp'].values.astype('datetime64[ns]').astype(np.float64)  # ns -> float
        ftm_ns = np.datetime64(ftm_time).astype('datetime64[ns]').astype(np.float64)
        dts = np.abs(times - ftm_ns)
        weights = 1 / (dts + 1e-9)  # 避免除零
        weights /= weights.sum()

        csi_data_all = []
        for row in csi_window['data']:
            if isinstance(row, str):
                csi_data_all.append(ast.literal_eval(row))
            else:
                csi_data_all.append(row)
        csi_data_all = np.array(csi_data_all, dtype=np.float32)
        csi_avg = np.average(csi_data_all, axis=0, weights=weights)
        """

        # 保存
        csi_list.append(csi_avg.tolist())
        rssi_list.append(float(nearest_row['rssi']))  # 取最近 CSI 的 RSSI
        rtt_list.append(float(ftm_row['rtt_raw (nsec)']))

    return csi_list, rssi_list, rtt_list


In [19]:
import os
import glob
import re
import numpy as np
import random

def load_all_data(data_dir, window_ms=100):
    """
    扫描 data_dir 下的 csi_data_*.csv 和 ftm_data_*.csv 文件
    自动对齐并合并，返回 csi_list, rssi_list, rtt_list, distance_list
    """
    csi_list_all, rssi_list_all, rtt_list_all, dist_list_all = [], [], [], []

    # 找到所有 csi 文件
    csi_files = glob.glob(os.path.join(data_dir, "csi_data_*.csv"))

    for csi_file in csi_files:
        # 提取距离 (比如 5.7 from csi_data_5.7m.csv)
        match = re.search(r"csi_data_(\d+(\.\d+)?)m\.csv", os.path.basename(csi_file))
        if not match:
            continue
        dist = float(match.group(1))

        # 找到对应的 ftm 文件
        ftm_file = os.path.join(data_dir, f"ftm_data_{dist}m.csv")
        if not os.path.exists(ftm_file):
            print(f"⚠️ 找不到 {ftm_file}, 跳过")
            continue

        # 调用你写的函数对齐
        csi_list, rssi_list, rtt_list = align_csi_ftm_from_csv(ftm_file, csi_file, window_ms)

        # 累加
        csi_list_all.extend(csi_list)
        rssi_list_all.extend(rssi_list)
        rtt_list_all.extend(rtt_list)
        dist_list_all.extend([dist] * len(csi_list))

    # 打乱顺序
    indices = list(range(len(csi_list_all)))
    random.shuffle(indices)

    csi_list_all = [csi_list_all[i] for i in indices]
    rssi_list_all = [rssi_list_all[i] for i in indices]
    rtt_list_all = [rtt_list_all[i] for i in indices]
    dist_list_all = [dist_list_all[i] for i in indices]

    return csi_list_all, rssi_list_all, rtt_list_all, dist_list_all

In [20]:
import numpy as np

def preprocess_csi(iq_list):
    """
    将原始 CSI I/Q 数据转换为 [幅度, 相位] 格式，并归一化
    参数：
        iq_list: list 或 np.ndarray，形如 [I0, Q0, I1, Q1, ...]
    返回：
        features: np.ndarray, shape = (num_subcarriers * 2,)
                  格式为 [amp0, phase0, amp1, phase1, ...]
    """
    iq_array = np.array(iq_list, dtype=np.float32)
    assert len(iq_array) % 2 == 0, "CSI 数据长度必须是偶数（I/Q 成对）"
    
    # 拆分 I / Q
    I = iq_array[0::2]
    Q = iq_array[1::2]
    
    # 计算幅度和相位
    amplitude = np.sqrt(I**2 + Q**2)
    phase = np.arctan2(Q, I)
    
    # 相位解缠（unwrap）
    phase = np.unwrap(phase)
    
    # 幅度归一化（0~1）
    if amplitude.max() > 0:
        amplitude = amplitude / amplitude.max()
    
    # 相位归一化到 [-1, 1]（原始范围大约是 -π ~ π）
    phase = phase / np.pi
    
    # 拼接为 [amp0, phase0, amp1, phase1, ...]
    features = np.empty(amplitude.size * 2, dtype=np.float32)
    features[0::2] = amplitude
    features[1::2] = phase
    
    return features


In [21]:
csi_list, rssi_list, rtt_list, dist_list = load_all_data("data")
csi_test, rssi_test, rtt_test, dist_test = load_all_data("data/test_data")

print("总样本数:", len(csi_list))
print("第一个样本: ", csi_list[0][:10])  # 只打印前10个元素
print("对应 RSSI:", rssi_list[0])
print("对应 RTT:", rtt_list[0])
print("对应距离:", dist_list[0])

总样本数: 3903
第一个样本:  [0.0, 0.0, 9.0, 11.0, 9.0, 10.0, 9.0, 10.0, 9.0, 10.0]
对应 RSSI: -40.0
对应 RTT: 15.0
对应距离: 1.8


In [22]:
class DistanceDataset:
    def __init__(self, csi_list, rssi_list, rtt_list, distance_list,
                 rssi_range=(-100, 0), rtt_max=300.0, dist_max=30.0):
        """
        csi_list: list of list, 每个样本的原始 I/Q 数据
        rssi_list: list of float, 单位 dBm
        rtt_list: list of float, 单位 ns
        distance_list: list of float, 单位 m
        """
        self.rssi_min, self.rssi_max = rssi_range
        self.rtt_max = rtt_max
        self.dist_max = dist_max

        # 预处理并保存
        self.csi_data = [preprocess_csi(csi) for csi in csi_list]
        self.rssi_data = [self.normalize_rssi(r) for r in rssi_list]
        self.rtt_data = [self.normalize_rtt(t) for t in rtt_list]
        self.distance_data = [self.normalize_distance(d) for d in distance_list]
        
    def __len__(self):
        return len(self.csi_data)
    
    def __getitem__(self, idx):
        csi = torch.tensor(self.csi_data[idx], dtype=torch.float32)
        rssi = torch.tensor([self.rssi_data[idx]], dtype=torch.float32)
        rtt = torch.tensor([self.rtt_data[idx]], dtype=torch.float32)
        dist = torch.tensor([self.distance_data[idx]], dtype=torch.float32)
        return csi, rssi, rtt, dist

    def normalize_rssi(self, rssi):
        return (rssi - self.rssi_min) / (self.rssi_max - self.rssi_min)

    def normalize_rtt(self, rtt):
        return rtt / self.rtt_max

    def normalize_distance(self, dist):
        return dist / self.dist_max

    def denormalize_distance(self, norm_dist):
        return norm_dist * self.dist_max
    
dataset = DistanceDataset(csi_list, rssi_list, rtt_list, dist_list)
dataset_test = DistanceDataset(csi_test, rssi_test, rtt_test, dist_test)

In [23]:
import torch
import torch.nn as nn

class MLPDistanceModel(nn.Module):
    def __init__(self, csi_dim):
        super().__init__()
        # CSI 分支
        self.csi_branch = nn.Sequential(
            nn.Linear(csi_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        # RTT+RSSI 分支
        self.other_branch = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU()
        )
        # 融合
        self.fc = nn.Sequential(
            nn.Linear(32 + 16, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # 输出距离
        )

    def forward(self, csi, rtt, rssi):
        csi_feat = self.csi_branch(csi)
        other_feat = self.other_branch(torch.cat([rtt, rssi], dim=1))
        x = torch.cat([csi_feat, other_feat], dim=1)
        return self.fc(x)

model = MLPDistanceModel(csi_dim=128)
print(model)

MLPDistanceModel(
  (csi_branch): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (other_branch): Sequential(
    (0): Linear(in_features=2, out_features=16, bias=True)
    (1): ReLU()
  )
  (fc): Sequential(
    (0): Linear(in_features=48, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [24]:
import torch.optim as optim

model = MLPDistanceModel(csi_dim=len(dataset.csi_data[0]))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

In [25]:
from torch.utils.data import DataLoader, random_split

dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)   # 80% 训练
val_size = dataset_size - train_size   # 20% 验证

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"训练集: {len(train_dataset)} 样本, 验证集: {len(val_dataset)} 样本")

训练集: 3122 样本, 验证集: 781 样本


In [26]:
epochs = 30

best_val_loss = float("inf")

for epoch in range(epochs):
    # ---- 训练 ----
    model.train()
    total_train_loss = 0.0
    for csi_batch, rssi_batch, rtt_batch, dist_batch in train_loader:
        pred = model(csi_batch, rtt_batch, rssi_batch)
        loss = criterion(pred, dist_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * csi_batch.size(0)

    avg_train_loss = total_train_loss / len(train_dataset)

    # ---- 验证 ----
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for csi_batch, rssi_batch, rtt_batch, dist_batch in val_loader:
            pred = model(csi_batch, rtt_batch, rssi_batch)
            loss = criterion(pred, dist_batch)
            total_val_loss += loss.item() * csi_batch.size(0)

    avg_val_loss = total_val_loss / len(val_dataset)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved new best model")

Epoch 1/30 | Train Loss: 0.001699 | Val Loss: 0.000575
Saved new best model
Epoch 2/30 | Train Loss: 0.000424 | Val Loss: 0.000266
Saved new best model
Epoch 3/30 | Train Loss: 0.000239 | Val Loss: 0.000165
Saved new best model
Epoch 4/30 | Train Loss: 0.000185 | Val Loss: 0.000160
Saved new best model
Epoch 5/30 | Train Loss: 0.000155 | Val Loss: 0.000155
Saved new best model
Epoch 6/30 | Train Loss: 0.000145 | Val Loss: 0.000158
Epoch 7/30 | Train Loss: 0.000130 | Val Loss: 0.000107
Saved new best model
Epoch 8/30 | Train Loss: 0.000105 | Val Loss: 0.000081
Saved new best model
Epoch 9/30 | Train Loss: 0.000091 | Val Loss: 0.000075
Saved new best model
Epoch 10/30 | Train Loss: 0.000089 | Val Loss: 0.000083
Epoch 11/30 | Train Loss: 0.000092 | Val Loss: 0.000082
Epoch 12/30 | Train Loss: 0.000081 | Val Loss: 0.000081
Epoch 13/30 | Train Loss: 0.000071 | Val Loss: 0.000061
Saved new best model
Epoch 14/30 | Train Loss: 0.000074 | Val Loss: 0.000063
Epoch 15/30 | Train Loss: 0.000060 |

In [27]:
csi, rssi, rtt, dist = dataset[16]
model.eval()
with torch.no_grad():
    pred = model(csi.unsqueeze(0), rtt.unsqueeze(0), rssi.unsqueeze(0))
    pred_dist = dataset.denormalize_distance(pred.item())
    true_dist = dataset.denormalize_distance(dist.item())
    print(f"预测距离: {pred_dist:.3f} m, 真实距离: {true_dist:.3f} m")

预测距离: 3.760 m, 真实距离: 3.600 m


In [35]:
csi_test, rssi_test, rtt_test, dist_test = load_all_data("data/test_data/6")
dataset_test = DistanceDataset(csi_test, rssi_test, rtt_test, dist_test)

In [36]:
def evaluate_accuracy(model, dataset_test, threshold=1.0):
    model.eval()
    correct = 0
    total = 0
    errors = []

    with torch.no_grad():
        for i in range(len(dataset_test)):
            csi, rssi, rtt, dist = dataset_test[i]

            # 增加 batch 维度
            pred = model(csi.unsqueeze(0), rtt.unsqueeze(0), rssi.unsqueeze(0))
            
            # 反归一化
            pred_dist = dataset_test.denormalize_distance(pred.item())
            true_dist = dataset_test.denormalize_distance(dist.item())
            
            print(pred_dist, true_dist)

            # 误差
            error = abs(pred_dist - true_dist)
            errors.append(error)

            if error <= threshold:
                correct += 1
            total += 1

    accuracy = correct / total * 100
    mean_error = sum(errors) / len(errors)

    print(f"Accuracy (±{threshold}m): {accuracy:.2f}%")
    print(f"Mean Absolute Error: {mean_error:.3f} m")

    return accuracy, mean_error

evaluate_accuracy(model, dataset_test, threshold=0.5)

9.115528464317322 6.000000089406967
9.34143751859665 6.000000089406967
9.70653623342514 6.000000089406967
10.212766528129578 6.000000089406967
9.525169730186462 6.000000089406967
9.177950620651245 6.000000089406967
7.7487534284591675 6.000000089406967
10.128283202648163 6.000000089406967
9.057330787181854 6.000000089406967
9.131213128566742 6.000000089406967
10.637229681015015 6.000000089406967
9.07920241355896 6.000000089406967
9.744073748588562 6.000000089406967
8.985536098480225 6.000000089406967
10.009282529354095 6.000000089406967
9.358559846878052 6.000000089406967
8.47656637430191 6.000000089406967
9.423184990882874 6.000000089406967
11.74360603094101 6.000000089406967
8.9888396859169 6.000000089406967
9.313310086727142 6.000000089406967
8.948552906513214 6.000000089406967
9.199142754077911 6.000000089406967
9.71667230129242 6.000000089406967
8.400234282016754 6.000000089406967
9.934431910514832 6.000000089406967
9.909581243991852 6.000000089406967
9.57688719034195 6.00000008940

(0.0, 3.347751254096944)